# Projection V2 — Local Training (GPU / CPU)

Notebook này thực thi huấn luyện mô hình **Projection V2** (mạng residual MLP + local contrastive loss) trực tiếp trên máy **local** (hỗ trợ GPU CUDA hoặc CPU), sử dụng trực tiếp các file input và embedding cache local mà không cần phải chờ hàng đợi Kaggle.

### Quy trình và Đặc điểm:
- **Dữ liệu input**: Sử dụng trực tiếp từ repository local (`configs/projection_v2.json`, `data/`, `cache/embeddings/`) hoặc file bundle `bundles/projection-v2-training-inputs.zip`.
- **Thiết bị (Accelerator)**: Cấu hình mặc định trong `configs/projection_v2.json` là `"device": "auto"`. Môi trường sẽ tự động sử dụng GPU (`cuda`) nếu có sẵn, hoặc tự động fallback CPU nếu không có CUDA GPU mà không làm thay đổi checksum/fingerprint của config.
- **Đầu ra**:
  - Ghi weights, checkpoint và manifest tại `artifacts/projection_v2/`
  - Cập nhật `artifacts/projection_v2/training_selection.json`
  - Đóng gói xuất bundle `bundles/projection-v2-heads.zip` hoàn chỉnh để dùng ngay cho Bước 2 và Bước 3 của `full-pipeline.ipynb`.

In [1]:
from pathlib import Path
import os, sys, json, torch

# Đảm bảo tính tất định (determinism) cho PyTorch
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Tự động phát hiện thư mục gốc GRACE-H1-SecurityProjection (hỗ trợ cả Windows và WSL)
candidates = ([Path(os.environ['GRACE_H1_ROOT'])] if os.environ.get('GRACE_H1_ROOT') else [])
candidates += [
    Path.cwd(),
    *Path.cwd().parents,
    Path('D:/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection'),
    Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection')
]
ROOT = next((p.resolve() for p in candidates if (p / 'grace_h1').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Không tìm thấy thư mục dự án GRACE-H1-SecurityProjection. Vui lòng set GRACE_H1_ROOT.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project ROOT: {ROOT}")

# Kiểm tra phần cứng khả dụng tại runtime
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ CUDA GPU phát hiện: {gpu_name}")
    runtime_device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("✅ Apple Silicon MPS phát hiện")
    runtime_device = 'mps'
else:
    print("ℹ️ Không có CUDA GPU được PyTorch nhận diện -> Runtime tự động chạy trên CPU")
    print("   (Mô hình MLP 256-dim huấn luyện trên cache 18k vector rất nhanh, ~1-2 phút)")
    runtime_device = 'cpu'

print(f"Runtime execution device: {runtime_device}")

RESUME_ZIP = None   # Có thể gán đường dẫn checkpoint zip nếu muốn tiếp tục lượt trước
EXTRA_SEEDS = []    # Ví dụ: [43, 44] để kiểm tra tính tái lập (replication)

Project ROOT: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection
ℹ️ Không có CUDA GPU được PyTorch nhận diện -> Runtime tự động chạy trên CPU
   (Mô hình MLP 256-dim huấn luyện trên cache 18k vector rất nhanh, ~1-2 phút)
Runtime execution device: cpu


In [2]:
from grace_h1.experiment_v2 import load_config, validate_config, training_identity
from grace_h1.io import read_json

# Đường dẫn các tài nguyên local
CONFIG_PATH = ROOT / 'configs/projection_v2.json'
DATA_MANIFEST = ROOT / 'data/manifest.json'
INPUT_ZIP = ROOT / 'bundles/projection-v2-training-inputs.zip'

print("Kiểm tra tính sẵn sàng của các input local:")
print(f" - Config file:       {CONFIG_PATH.exists()} ({CONFIG_PATH})")
print(f" - Data manifest:     {DATA_MANIFEST.exists()} ({DATA_MANIFEST})")
print(f" - Training inputs:   {INPUT_ZIP.exists()} ({INPUT_ZIP})")

# Đọc cấu hình chuẩn từ configs/projection_v2.json (giữ nguyên device='auto' để đồng bộ hash)
config = load_config(ROOT)
validate_config(config)
print(f"\nTraining config fingerprint: {training_identity(config)}")
print("\n--- Cấu hình Projection V2 đã nạp ---")
print(json.dumps(config, indent=2))

Kiểm tra tính sẵn sàng của các input local:
 - Config file:       True (/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/configs/projection_v2.json)
 - Data manifest:     True (/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/data/manifest.json)
 - Training inputs:   True (/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/bundles/projection-v2-training-inputs.zip)

Training config fingerprint: 9a51f373e5bb042f99c5f5e4a4f4e66575e2d85e4c4184614cc105476bb5eff8

--- Cấu hình Projection V2 đã nạp ---
{
  "datasets": [
    "reveal"
  ],
  "splits": [
    "validation"
  ],
  "objectives": [
    "residual_ce",
    "residual_local"
  ],
  "llm_arms": [
    "baseline_whitening",
    "h1_residual_local"
  ],
  "training": {
    "dimension": 256,
    "hidden_dimension": 256,
    "residual_radius": 0.25,
    "dropout": 0.1,
    "epochs": 30,
    "batch_size": 128,
    "learning_rate": 0.0003,
    "weight_decay": 0.001,
    "temperature": 0.1,
    "margin": 0.2,
    "local_weight": 0.1,
 

## Huấn luyện và Tuyển chọn Head (Model Selection)

Tiến hành huấn luyện các mục tiêu trong config (mặc định: `residual_ce` và `residual_local` trên dataset `reveal`).
- Quá trình huấn luyện đọc trực tiếp embedding cache đã chuẩn bị từ trước.
- Mỗi epoch kiểm tra macro label P@5 trên validation và overlap với teacher.
- Checkpoint tốt nhất và manifest tuyển chọn được lưu tại `artifacts/projection_v2/`.

In [3]:
import copy
from grace_h1.experiment_v2 import train_all
from grace_h1.portable_v2 import create_bundle
from grace_h1.io import write_json

print(f"Bắt đầu huấn luyện Projection V2 (device='{config['training']['device']}', runtime={runtime_device})...")
try:
    selected = train_all(ROOT, config)
    replications = {str(config['training']['seed']): selected}
    for seed in EXTRA_SEEDS:
        print(f"\n--- Huấn luyện bổ sung với seed={seed} ---")
        replica = copy.deepcopy(config)
        replica['training']['seed'] = int(seed)
        replications[str(seed)] = train_all(ROOT, replica)
    if EXTRA_SEEDS:
        write_json(ROOT / 'artifacts/projection_v2/replications.json', replications)
        selected = train_all(ROOT, config)  # Khôi phục lựa chọn cho seed chính
finally:
    # Luôn đóng gói checkpoint đề phòng ngắt quãng giữa chừng
    checkpoint_path = ROOT / 'bundles/projection-v2-training-checkpoint.zip'
    if (ROOT / 'artifacts/projection_v2').exists():
        chk = create_bundle(ROOT, checkpoint_path, 'training_checkpoint')
        print(f"Checkpoint archive saved: {chk['path']} ({chk['bytes']:,} bytes)")

print("\n--- TỔNG KẾT KẾT QUẢ TUYỂN CHỌN HEAD ---")
for dataset_name, heads in selected['heads'].items():
    for obj_name, h in heads.items():
        print(f"[{dataset_name}] {obj_name:<16} | Status: {h['selection_status']} | Best: {h['best']}")

Bắt đầu huấn luyện Projection V2 (device='auto', runtime=cpu)...
reveal/residual_local epoch 4: loss=0.5280, macro P@5=0.6345, overlap=0.932, accepted=False
reveal/residual_local epoch 5: loss=0.4882, macro P@5=0.6303, overlap=0.930, accepted=False
reveal/residual_local epoch 6: loss=0.4557, macro P@5=0.6343, overlap=0.932, accepted=False
reveal/residual_local epoch 7: loss=0.4393, macro P@5=0.6320, overlap=0.931, accepted=False
reveal/residual_local epoch 8: loss=0.4176, macro P@5=0.6337, overlap=0.931, accepted=False
reveal/residual_local epoch 9: loss=0.4005, macro P@5=0.6291, overlap=0.933, accepted=False
reveal/residual_local epoch 10: loss=0.3887, macro P@5=0.6300, overlap=0.931, accepted=False
Checkpoint archive saved: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/bundles/projection-v2-training-checkpoint.zip (23,200,938 bytes)

--- TỔNG KẾT KẾT QUẢ TUYỂN CHỌN HEAD ---
[reveal] residual_ce      | Status: validation_proxy_gain | Best: {'epoch': 4, 'retrieval': {'classifier

## Đóng gói xuất bundle `projection-v2-heads.zip`

Tạo file `bundles/projection-v2-heads.zip` chứa đầy đủ model heads đã tuyển chọn.
Sau khi cell này hoàn thành, bạn có thể quay lại chạy tiếp Bước 2 / Bước 3 trong `full-pipeline.ipynb`!

In [4]:
from grace_h1.portable_v2 import create_bundle

HEADS_ZIP = ROOT / 'bundles/projection-v2-heads.zip'
result = create_bundle(ROOT, HEADS_ZIP, 'heads')

print("🎉 ĐÃ TẠO THÀNH CÔNG BUNDLE HEADS LOCAL!")
print(f" - Path:    {result['path']}")
print(f" - Files:   {result['files']}")
print(f" - Bytes:   {result['bytes']:,} bytes")
print(f" - SHA256:  {result['sha256']}")
print("\n👉 Bạn có thể mở lại 'full-pipeline.ipynb', Bước 1 đã hoàn thành đầy đủ!")

🎉 ĐÃ TẠO THÀNH CÔNG BUNDLE HEADS LOCAL!
 - Path:    /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/bundles/projection-v2-heads.zip
 - Files:   23
 - Bytes:   23,200,995 bytes
 - SHA256:  4497acf164548be18ab022253caa257569abb77e9457b2c91f0767e178f880dd

👉 Bạn có thể mở lại 'full-pipeline.ipynb', Bước 1 đã hoàn thành đầy đủ!
